In [177]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="Bangtan@army1"
)
cursor = conn.cursor()
print("Connected to MySQL")

Connected to MySQL


In [178]:
cursor.execute("CREATE DATABASE IF NOT EXISTS healthcare_db")
cursor.execute("USE healthcare_db")
conn.database = "healthcare_db"
print("Database ready")

Database ready


In [179]:
import pandas as pd
import hashlib

df_health   = pd.read_csv(r"C:\Users\Ananya Lakkaraju\Downloads\Healthcare-Analytics-Project\notebooks\Case Study 1 - Healthcare Analysis\cleanedHealthcare_data.csv")
df_disease  = pd.read_csv(r"C:\Users\Ananya Lakkaraju\Downloads\Healthcare-Analytics-Project\notebooks\Case Study 2 - Diabetes Prediction Analysis\cleaned_disease_prediction.csv")
df_hospital = pd.read_csv(r"C:\Users\Ananya Lakkaraju\Downloads\Healthcare-Analytics-Project\notebooks\Case Study 3 - Hospital Management Analysis\cleanedHospital_data.csv")

print(df_health.shape, df_disease.shape, df_hospital.shape)


(54966, 15) (1000, 22) (200, 9)


In [180]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS healthcare (
    patient_id       VARCHAR(20) PRIMARY KEY,
    name             VARCHAR(100),
    age              INT,
    gender           VARCHAR(10),
    blood_type       VARCHAR(5),
    medical_condition VARCHAR(100),
    date_of_admission DATE,
    doctor           VARCHAR(100),
    hospital         VARCHAR(100),
    insurance_provider VARCHAR(50),
    billing_amount   FLOAT,
    room_number      INT,
    admission_type   VARCHAR(30),
    discharge_date   DATE,
    medication       VARCHAR(100),
    test_results     VARCHAR(50)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS disease_prediction (
    id               INT AUTO_INCREMENT PRIMARY KEY,
    patient_id       VARCHAR(20),
    age              INT,
    gender           VARCHAR(10),
    blood_pressure   VARCHAR(20),
    cholesterol      VARCHAR(20),
    glucose          VARCHAR(20),
    smoking          VARCHAR(5),
    alcohol_consumption VARCHAR(5),
    exercise         VARCHAR(5),
    bmi              FLOAT,
    family_history   VARCHAR(5),
    heart_disease    INT,
    diabetes         INT,
    stroke           INT,
    kidney_disease   INT,
    cancer           INT,
    alzheimers_disease INT,
    copd             INT,
    liver_disease    INT,
    parkinsons_disease INT,
    tuberculosis     INT,
    age_group        VARCHAR(20),
    bmi_category     VARCHAR(20),
    FOREIGN KEY (patient_id) REFERENCES healthcare(patient_id)
        ON DELETE SET NULL ON UPDATE CASCADE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS hospital_management (
    appointment_id   VARCHAR(50) PRIMARY KEY,
    patient_id       VARCHAR(20),
    doctor_id        VARCHAR(50),
    appointment_date DATE,
    appointment_time TIME,
    reason_for_visit VARCHAR(100),
    status           VARCHAR(30),
    month            VARCHAR(20),
    day              VARCHAR(20),
    FOREIGN KEY (patient_id) REFERENCES healthcare(patient_id)
        ON DELETE SET NULL ON UPDATE CASCADE
)
""")

conn.commit()
print("All tables created")

All tables created


In [181]:
# Add this as Cell 4.5 — run between table creation and insertion

def normalize_columns(df):
    """Lowercase, strip spaces, replace spaces with underscores."""
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

df_health   = normalize_columns(df_health)
df_disease  = normalize_columns(df_disease)
df_hospital = normalize_columns(df_hospital)

# Then update the column references in Cell 5 to match normalized names
# Check what they look like now:
print("health:  ", df_health.columns.tolist())
print("disease: ", df_disease.columns.tolist())
print("hospital:", df_hospital.columns.tolist())

health:   ['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']
disease:  ['age', 'gender', 'blood_pressure', 'cholesterol', 'glucose', 'smoking', 'alcohol_consumption', 'exercise', 'bmi', 'family_history', 'heart_disease', 'diabetes', 'stroke', 'kidney_disease', 'cancer', "alzheimer's_disease", 'copd', 'liver_disease', "parkinson's_disease", 'tuberculosis', 'age_group', 'bmi_category']
hospital: ['appointment_id', 'patient_id', 'doctor_id', 'appointment_date', 'appointment_time', 'reason_for_visit', 'status', 'month', 'day']


In [182]:
# Diagnostic + fix cell — run before Cell 5

print("patient_id in health: ", 'patient_id' in df_health.columns)
print("patient_id in disease:", 'patient_id' in df_disease.columns)

# If both are False, we generate them now
import hashlib

def generate_patient_id(row):
    """Generate consistent ID from name + age + gender."""
    key = f"{str(row.get('name',''))}_{str(row.get('age',''))}_{str(row.get('gender',''))}"
    return hashlib.md5(key.encode()).hexdigest()[:10]

if 'patient_id' not in df_health.columns:
    df_health['patient_id'] = df_health.apply(generate_patient_id, axis=1)
    print("Generated patient_id for health")

if 'patient_id' not in df_disease.columns:
    df_disease['patient_id'] = df_disease.apply(generate_patient_id, axis=1)
    print("Generated patient_id for disease")

print("\nSample health patient_ids: ", df_health['patient_id'].head(3).tolist())
print("Sample disease patient_ids:", df_disease['patient_id'].head(3).tolist())

patient_id in health:  False
patient_id in disease: False
Generated patient_id for health
Generated patient_id for disease

Sample health patient_ids:  ['a023eaa979', 'f8275577d2', '7dc5fb6cae']
Sample disease patient_ids: ['4c2f7f1feb', 'a74b700739', '93035ba76c']


In [183]:
from datetime import datetime

def safe_date(val):
    if pd.isna(val):
        return None
    try:
        return pd.to_datetime(val).strftime('%Y-%m-%d')
    except:
        return None

def safe_time(val):
    if pd.isna(val):
        return None
    try:
        return str(pd.to_datetime(val, format='%H:%M:%S').time())
    except:
        try:
            return str(pd.to_datetime(val).time())
        except:
            return None

def safe_val(val):
    try:
        if pd.isna(val):
            return None
    except:
        pass
    return val

# ── healthcare ──────────────────────────────────────────────────────────────
insert_health = """
    INSERT IGNORE INTO healthcare
    (patient_id, name, age, gender, blood_type, medical_condition,
     date_of_admission, doctor, hospital, insurance_provider,
     billing_amount, room_number, admission_type, discharge_date,
     medication, test_results)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

health_rows = []
for _, r in df_health.iterrows():
    health_rows.append((
        safe_val(r['patient_id']),
        safe_val(r['name']),
        safe_val(r['age']),
        safe_val(r['gender']),
        safe_val(r['blood_type']),
        safe_val(r['medical_condition']),
        safe_date(r['date_of_admission']),
        safe_val(r['doctor']),
        safe_val(r['hospital']),
        safe_val(r['insurance_provider']),
        safe_val(r['billing_amount']),
        safe_val(r['room_number']),
        safe_val(r['admission_type']),
        safe_date(r['discharge_date']),
        safe_val(r['medication']),
        safe_val(r['test_results'])
    ))

cursor.executemany(insert_health, health_rows)
conn.commit()
print(f"healthcare: {cursor.rowcount} rows inserted")

# ── disease_prediction ──────────────────────────────────────────────────────
insert_disease = """
    INSERT INTO disease_prediction
    (patient_id, age, gender, blood_pressure, cholesterol, glucose,
     smoking, alcohol_consumption, exercise, bmi, family_history,
     heart_disease, diabetes, stroke, kidney_disease, cancer,
     alzheimers_disease, copd, liver_disease, parkinsons_disease,
     tuberculosis, age_group, bmi_category)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

disease_rows = []
for _, r in df_disease.iterrows():
    disease_rows.append((
        safe_val(r['patient_id']),
        safe_val(r['age']),
        safe_val(r['gender']),
        safe_val(r['blood_pressure']),
        safe_val(r['cholesterol']),
        safe_val(r['glucose']),
        safe_val(r['smoking']),
        safe_val(r['alcohol_consumption']),
        safe_val(r['exercise']),
        safe_val(r['bmi']),
        safe_val(r['family_history']),
        safe_val(r['heart_disease']),
        safe_val(r['diabetes']),
        safe_val(r['stroke']),
        safe_val(r['kidney_disease']),
        safe_val(r['cancer']),
        safe_val(r["alzheimer's_disease"]),
        safe_val(r['copd']),
        safe_val(r['liver_disease']),
        safe_val(r["parkinson's_disease"]),
        safe_val(r['tuberculosis']),
        safe_val(r['age_group']),
        safe_val(r['bmi_category'])
    ))

cursor.executemany(insert_disease, disease_rows)
conn.commit()
print(f"disease_prediction: {cursor.rowcount} rows inserted")

# ── hospital_management ─────────────────────────────────────────────────────
insert_hospital = """
    INSERT IGNORE INTO hospital_management
    (appointment_id, patient_id, doctor_id, appointment_date,
     appointment_time, reason_for_visit, status, month, day)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

hospital_rows = []
for _, r in df_hospital.iterrows():
    hospital_rows.append((
        safe_val(r['appointment_id']),
        safe_val(r['patient_id']),
        safe_val(r['doctor_id']),
        safe_date(r['appointment_date']),
        safe_time(r['appointment_time']),
        safe_val(r['reason_for_visit']),
        safe_val(r['status']),
        safe_val(r['month']),
        safe_val(r['day'])
    ))

cursor.executemany(insert_hospital, hospital_rows)
conn.commit()
print(f"hospital_management: {cursor.rowcount} rows inserted")

healthcare: 0 rows inserted
disease_prediction: 1000 rows inserted
hospital_management: 0 rows inserted


In [184]:
# Drop and recreate tables without foreign key constraints
# (These are independent datasets — FK constraints don't apply)

cursor.execute("USE healthcare_db")

cursor.execute("DROP TABLE IF EXISTS disease_prediction")
cursor.execute("DROP TABLE IF EXISTS hospital_management")
cursor.execute("DROP TABLE IF EXISTS healthcare")

cursor.execute("""
CREATE TABLE healthcare (
    patient_id        VARCHAR(20) PRIMARY KEY,
    name              VARCHAR(100),
    age               INT,
    gender            VARCHAR(10),
    blood_type        VARCHAR(5),
    medical_condition VARCHAR(100),
    date_of_admission DATE,
    doctor            VARCHAR(100),
    hospital          VARCHAR(100),
    insurance_provider VARCHAR(50),
    billing_amount    FLOAT,
    room_number       INT,
    admission_type    VARCHAR(30),
    discharge_date    DATE,
    medication        VARCHAR(100),
    test_results      VARCHAR(50)
)
""")

cursor.execute("""
CREATE TABLE disease_prediction (
    id                  INT AUTO_INCREMENT PRIMARY KEY,
    patient_id          VARCHAR(20),
    age                 INT,
    gender              VARCHAR(10),
    blood_pressure      VARCHAR(20),
    cholesterol         VARCHAR(20),
    glucose             VARCHAR(20),
    smoking             VARCHAR(5),
    alcohol_consumption VARCHAR(5),
    exercise            VARCHAR(5),
    bmi                 FLOAT,
    family_history      VARCHAR(5),
    heart_disease       INT,
    diabetes            INT,
    stroke              INT,
    kidney_disease      INT,
    cancer              INT,
    alzheimers_disease  INT,
    copd                INT,
    liver_disease       INT,
    parkinsons_disease  INT,
    tuberculosis        INT,
    age_group           VARCHAR(20),
    bmi_category        VARCHAR(20)
)
""")

cursor.execute("""
CREATE TABLE hospital_management (
    appointment_id   VARCHAR(50) PRIMARY KEY,
    patient_id       VARCHAR(20),
    doctor_id        VARCHAR(50),
    appointment_date DATE,
    appointment_time TIME,
    reason_for_visit VARCHAR(100),
    status           VARCHAR(30),
    month            VARCHAR(20),
    day              VARCHAR(20)
)
""")

conn.commit()
print("Tables recreated without FK constraints — ready to insert")

Tables recreated without FK constraints — ready to insert


In [185]:
from datetime import datetime

def safe_date(val):
    if pd.isna(val):
        return None
    try:
        return pd.to_datetime(val).strftime('%Y-%m-%d')
    except:
        return None

def safe_time(val):
    if pd.isna(val):
        return None
    try:
        return str(pd.to_datetime(val, format='%H:%M:%S').time())
    except:
        try:
            return str(pd.to_datetime(val).time())
        except:
            return None

def safe_val(val):
    try:
        if pd.isna(val):
            return None
    except:
        pass
    return val

# ── healthcare ──────────────────────────────────────────────────────────────
insert_health = """
    INSERT IGNORE INTO healthcare
    (patient_id, name, age, gender, blood_type, medical_condition,
     date_of_admission, doctor, hospital, insurance_provider,
     billing_amount, room_number, admission_type, discharge_date,
     medication, test_results)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

health_rows = []
for _, r in df_health.iterrows():
    health_rows.append((
        safe_val(r['patient_id']),
        safe_val(r['name']),
        safe_val(r['age']),
        safe_val(r['gender']),
        safe_val(r['blood_type']),
        safe_val(r['medical_condition']),
        safe_date(r['date_of_admission']),
        safe_val(r['doctor']),
        safe_val(r['hospital']),
        safe_val(r['insurance_provider']),
        safe_val(r['billing_amount']),
        safe_val(r['room_number']),
        safe_val(r['admission_type']),
        safe_date(r['discharge_date']),
        safe_val(r['medication']),
        safe_val(r['test_results'])
    ))

cursor.executemany(insert_health, health_rows)
conn.commit()
print(f"healthcare: {cursor.rowcount} rows inserted")

# ── disease_prediction ──────────────────────────────────────────────────────
insert_disease = """
    INSERT INTO disease_prediction
    (patient_id, age, gender, blood_pressure, cholesterol, glucose,
     smoking, alcohol_consumption, exercise, bmi, family_history,
     heart_disease, diabetes, stroke, kidney_disease, cancer,
     alzheimers_disease, copd, liver_disease, parkinsons_disease,
     tuberculosis, age_group, bmi_category)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

disease_rows = []
for _, r in df_disease.iterrows():
    disease_rows.append((
        safe_val(r['patient_id']),
        safe_val(r['age']),
        safe_val(r['gender']),
        safe_val(r['blood_pressure']),
        safe_val(r['cholesterol']),
        safe_val(r['glucose']),
        safe_val(r['smoking']),
        safe_val(r['alcohol_consumption']),
        safe_val(r['exercise']),
        safe_val(r['bmi']),
        safe_val(r['family_history']),
        safe_val(r['heart_disease']),
        safe_val(r['diabetes']),
        safe_val(r['stroke']),
        safe_val(r['kidney_disease']),
        safe_val(r['cancer']),
        safe_val(r["alzheimer's_disease"]),
        safe_val(r['copd']),
        safe_val(r['liver_disease']),
        safe_val(r["parkinson's_disease"]),
        safe_val(r['tuberculosis']),
        safe_val(r['age_group']),
        safe_val(r['bmi_category'])
    ))

cursor.executemany(insert_disease, disease_rows)
conn.commit()
print(f"disease_prediction: {cursor.rowcount} rows inserted")

# ── hospital_management ─────────────────────────────────────────────────────
insert_hospital = """
    INSERT IGNORE INTO hospital_management
    (appointment_id, patient_id, doctor_id, appointment_date,
     appointment_time, reason_for_visit, status, month, day)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

hospital_rows = []
for _, r in df_hospital.iterrows():
    hospital_rows.append((
        safe_val(r['appointment_id']),
        safe_val(r['patient_id']),
        safe_val(r['doctor_id']),
        safe_date(r['appointment_date']),
        safe_time(r['appointment_time']),
        safe_val(r['reason_for_visit']),
        safe_val(r['status']),
        safe_val(r['month']),
        safe_val(r['day'])
    ))

cursor.executemany(insert_hospital, hospital_rows)
conn.commit()
print(f"hospital_management: {cursor.rowcount} rows inserted")

healthcare: 54966 rows inserted
disease_prediction: 1000 rows inserted
hospital_management: 200 rows inserted


In [186]:
for table in ['healthcare', 'disease_prediction', 'hospital_management']:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cursor.fetchone()[0]} rows")

healthcare: 54966 rows
disease_prediction: 1000 rows
hospital_management: 200 rows


In [187]:
def run_query(sql, label=""):
    df = pd.read_sql(sql, conn)
    if label:
        print(f"\n{'─'*60}")
        print(f"  {label}")
        print(f"{'─'*60}")
    return df

In [188]:
# Q1 — Billing by medical condition
run_query("""
    SELECT medical_condition,
           COUNT(*)                        AS total_patients,
           ROUND(AVG(billing_amount), 2)   AS avg_billing,
           ROUND(MIN(billing_amount), 2)   AS min_billing,
           ROUND(MAX(billing_amount), 2)   AS max_billing
    FROM healthcare
    GROUP BY medical_condition
    ORDER BY avg_billing DESC
""", "Billing by Medical Condition")


────────────────────────────────────────────────────────────
  Billing by Medical Condition
────────────────────────────────────────────────────────────


C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1118101782.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,medical_condition,total_patients,avg_billing,min_billing,max_billing
0,Obesity,9146,25804.36,-1310.27,52024.73
1,Diabetes,9216,25660.48,-1316.62,52211.85
2,Asthma,9095,25633.46,-1520.42,52181.84
3,Arthritis,9218,25511.78,-1130.00,52170.04
4,Hypertension,9151,25503.06,-1660.01,52764.28
5,Cancer,9140,25152.32,-2008.49,52373.03


In [189]:
# Q2 — No-show rate by reason for visit
run_query("""
    SELECT reason_for_visit,
           COUNT(*)                                              AS total_appointments,
           SUM(status = 'No-Show')                              AS no_shows,
           ROUND(SUM(status = 'No-Show') * 100.0 / COUNT(*), 1) AS no_show_pct
    FROM hospital_management
    GROUP BY reason_for_visit
    ORDER BY no_show_pct DESC
""", "No-Show Rate by Reason for Visit")


────────────────────────────────────────────────────────────
  No-Show Rate by Reason for Visit
────────────────────────────────────────────────────────────


C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1118101782.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,reason_for_visit,total_appointments,no_shows,no_show_pct
0,Therapy,42,15.0,35.7
1,Emergency,29,10.0,34.5
2,Consultation,43,11.0,25.6
3,Checkup,45,10.0,22.2
4,Follow-Up,41,6.0,14.6


In [190]:
# Q3 — Disease burden by BMI category
run_query("""
    SELECT bmi_category,
           COUNT(DISTINCT patient_id)      AS patient_count,
           ROUND(AVG(bmi), 2)              AS avg_bmi,
           SUM(heart_disease)              AS heart_disease_cases,
           SUM(diabetes)                   AS diabetes_cases,
           SUM(stroke)                     AS stroke_cases,
           SUM(cancer)                     AS cancer_cases
    FROM disease_prediction
    GROUP BY bmi_category
    ORDER BY avg_bmi
""", "Disease Burden by BMI Category")


────────────────────────────────────────────────────────────
  Disease Burden by BMI Category
────────────────────────────────────────────────────────────


C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1118101782.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,bmi_category,patient_count,avg_bmi,heart_disease_cases,diabetes_cases,stroke_cases,cancer_cases
0,Normal,117,21.78,75.0,56.0,35.0,32.0
1,Overweight,116,27.33,65.0,55.0,40.0,27.0
2,Obese,136,35.16,113.0,75.0,59.0,41.0


In [191]:
# Q4 — High-risk patients: combined profile (no JOIN, row-position merge)
df_health_db   = pd.read_sql("SELECT * FROM healthcare", conn)
df_disease_db  = pd.read_sql("SELECT * FROM disease_prediction", conn)

# Merge side by side by row position — reset index to align cleanly
df_health_db  = df_health_db.reset_index(drop=True)
df_disease_db = df_disease_db.reset_index(drop=True)

# Take only the columns we need from each
df_combined = pd.concat([
    df_health_db[['patient_id', 'age', 'gender', 'medical_condition',
                  'admission_type', 'billing_amount', 'test_results']],
    df_disease_db[['bmi', 'bmi_category', 'blood_pressure',
                   'cholesterol', 'smoking', 'exercise',
                   'heart_disease', 'diabetes', 'stroke',
                   'kidney_disease', 'cancer']]
], axis=1)

# Calculate disease count per patient
df_combined['disease_count'] = (
    df_combined['heart_disease'] +
    df_combined['diabetes']      +
    df_combined['stroke']        +
    df_combined['kidney_disease']+
    df_combined['cancer']
)

# Round billing
df_combined['billing_amount'] = df_combined['billing_amount'].round(2)

# Sort and show top 20 high-risk
df_result = df_combined.sort_values(
    ['disease_count', 'billing_amount'],
    ascending=[False, False]
).head(20)

print("\n────────────────────────────────────────────────────────────")
print("  High-Risk Patients: Combined Profile")
print("────────────────────────────────────────────────────────────")
df_result

C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1178292436.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_health_db   = pd.read_sql("SELECT * FROM healthcare", conn)



────────────────────────────────────────────────────────────
  High-Risk Patients: Combined Profile
────────────────────────────────────────────────────────────


C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1178292436.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_disease_db  = pd.read_sql("SELECT * FROM disease_prediction", conn)


,patient_id,age,gender,medical_condition,admission_type,billing_amount,test_results,bmi,bmi_category,blood_pressure,cholesterol,smoking,exercise,heart_disease,diabetes,stroke,kidney_disease,cancer,disease_count
246,014a10b6c6,48,Female,Hypertension,Emergency,31338.90,Normal,25.3886,Overweight,Low,Normal,No,No,1.0,1.0,0.0,1.0,1.0,4.0
405,020f886d76,57,Female,Diabetes,Emergency,9934.77,Abnormal,28.1825,Overweight,High,Normal,No,Yes,1.0,1.0,1.0,1.0,0.0,4.0
973,04b932b5f7,55,Female,Hypertension,Urgent,7419.70,Normal,32.3978,Obese,Normal,Normal,No,Yes,1.0,1.0,0.0,1.0,1.0,4.0
784,03d4c7c71e,71,Male,Hypertension,Urgent,50737.40,Normal,25.0192,Overweight,Normal,High,Yes,No,1.0,1.0,0.0,1.0,0.0,3.0
298,01892fa6df,34,Male,Hypertension,Elective,42299.90,Normal,21.5358,Normal,Low,High,Yes,No,1.0,0.0,0.0,1.0,1.0,3.0
843,042167421d,78,Male,Arthritis,Urgent,42058.60,Abnormal,37.1616,Obese,Low,Normal,No,Yes,1.0,1.0,1.0,0.0,0.0,3.0
825,04035bf7e0,74,Female,Asthma,Elective,39618.80,Normal,28.9097,Overweight,High,High,Yes,No,1.0,0.0,1.0,0.0,1.0,3.0
665,03476546bc,69,Male,Diabetes,Elective,39354.10,Abnormal,26.9657,Overweight,High,High,No,No,1.0,0.0,1.0,0.0,1.0,3.0
3,000cf65a8c,77,Male,Cancer,Emergency,32440.30,Inconclusive,21.8063,Normal,High,High,No,Yes,0.0,1.0,1.0,0.0,1.0,3.0
21,0027eaed8c,66,Male,Diabetes,Urgent,27888.00,Inconclusive,29.2513,Overweight,High,High,No,No,1.0,0.0,0.0,1.0,1.0,3.0


In [192]:
# Q5 — Monthly appointment trends
run_query("""
    SELECT month,
           COUNT(appointment_id)           AS total_appointments,
           SUM(status = 'Scheduled')       AS scheduled,
           SUM(status = 'No-Show')         AS no_show,
           SUM(status = 'Completed')       AS completed
    FROM hospital_management
    GROUP BY month
    ORDER BY FIELD(month,
        'January','February','March','April','May','June',
        'July','August','September','October','November','December')
""", "Monthly Appointment Trends")


────────────────────────────────────────────────────────────
  Monthly Appointment Trends
────────────────────────────────────────────────────────────


C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1118101782.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,month,total_appointments,scheduled,no_show,completed
0,January,20,5.0,8.0,1.0
1,February,14,2.0,3.0,6.0
2,March,19,8.0,3.0,4.0
3,April,25,4.0,5.0,9.0
4,May,19,2.0,5.0,6.0
5,June,18,6.0,6.0,4.0
6,July,16,3.0,4.0,5.0
7,August,15,6.0,3.0,3.0
8,September,11,3.0,3.0,2.0
9,October,14,4.0,5.0,0.0


In [193]:
# Q6 — Lifestyle factors vs disease incidence
run_query("""
    SELECT smoking,
           alcohol_consumption,
           COUNT(*)                            AS patients,
           ROUND(AVG(heart_disease) * 100, 1) AS heart_disease_pct,
           ROUND(AVG(diabetes) * 100, 1)      AS diabetes_pct,
           ROUND(AVG(stroke) * 100, 1)        AS stroke_pct,
           ROUND(AVG(bmi), 2)                 AS avg_bmi
    FROM disease_prediction
    GROUP BY smoking, alcohol_consumption
    ORDER BY heart_disease_pct DESC
""", "Lifestyle Factors vs Disease Incidence")

C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1118101782.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)



────────────────────────────────────────────────────────────
  Lifestyle Factors vs Disease Incidence
────────────────────────────────────────────────────────────


,smoking,alcohol_consumption,patients,heart_disease_pct,diabetes_pct,stroke_pct,avg_bmi
0,Yes,No,251,27.1,19.5,15.9,29.11
1,No,Yes,244,26.2,20.5,12.3,29.38
2,No,No,254,24.8,16.9,16.5,29.01
3,Yes,Yes,251,23.1,17.5,8.8,29.36


In [194]:
# Q7 — Insurance provider billing analysis
run_query("""
    SELECT insurance_provider,
           COUNT(*)                           AS total_claims,
           ROUND(AVG(billing_amount), 2)      AS avg_claim,
           ROUND(SUM(billing_amount), 2)      AS total_billed,
           SUM(admission_type = 'Emergency')  AS emergency_admissions,
           SUM(admission_type = 'Elective')   AS elective_admissions
    FROM healthcare
    GROUP BY insurance_provider
    ORDER BY total_billed DESC
""", "Insurance Provider Billing Summary")


────────────────────────────────────────────────────────────
  Insurance Provider Billing Summary
────────────────────────────────────────────────────────────


C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1118101782.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,insurance_provider,total_claims,avg_claim,total_billed,emergency_admissions,elective_admissions
0,Cigna,11139,25526.00,2.843341e+08,3638.0,3676.0
1,Medicare,11039,25628.32,2.829110e+08,3643.0,3637.0
2,Blue Cross,10952,25603.46,2.804091e+08,3597.0,3709.0
3,UnitedHealthcare,11014,25414.51,2.799154e+08,3583.0,3733.0
4,Aetna,10822,25549.69,2.764987e+08,3641.0,3718.0


In [195]:
# Q8 — Doctor workload + outcomes (three-table JOIN)
run_query("""
    SELECT h.doctor,
           COUNT(DISTINCT h.patient_id)         AS patients_treated,
           COUNT(DISTINCT hm.appointment_id)    AS total_appointments,
           ROUND(AVG(h.billing_amount), 2)      AS avg_billing,
           SUM(hm.status = 'No-Show')           AS no_shows,
           SUM(h.test_results = 'Normal')       AS normal_results,
           SUM(h.test_results = 'Abnormal')     AS abnormal_results,
           SUM(h.test_results = 'Inconclusive') AS inconclusive_results
    FROM healthcare h
    LEFT JOIN hospital_management hm ON h.patient_id = hm.patient_id
    GROUP BY h.doctor
    ORDER BY patients_treated DESC
    LIMIT 15
""", "Doctor Workload and Patient Outcomes")

C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\1118101782.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)



────────────────────────────────────────────────────────────
  Doctor Workload and Patient Outcomes
────────────────────────────────────────────────────────────


,doctor,patients_treated,total_appointments,avg_billing,no_shows,normal_results,abnormal_results,inconclusive_results
0,Michael Smith,27,0,29055.62,None,6.0,12.0,9.0
1,John Smith,22,0,27732.25,None,9.0,7.0,6.0
2,Robert Smith,21,0,29007.65,None,7.0,5.0,9.0
3,James Smith,20,0,23090.90,None,5.0,8.0,7.0
4,Michael Johnson,20,0,23040.95,None,13.0,5.0,2.0
5,Robert Johnson,19,0,27589.11,None,5.0,6.0,8.0
6,David Smith,19,0,24912.93,None,7.0,5.0,7.0
7,Michael Williams,18,0,17171.73,None,9.0,4.0,5.0
8,Christopher Smith,17,0,19929.48,None,4.0,7.0,6.0
9,Matthew Smith,17,0,24790.30,None,11.0,3.0,3.0


In [198]:
import os
import pandas as pd

output_dir = r"C:\Users\Ananya Lakkaraju\Downloads\Healthcare-Analytics-Project\powerbi_exports"
os.makedirs(output_dir, exist_ok=True)

# 1. Pure SQL Queries Dictionary
exports = {
    "billing_by_condition": """
        SELECT medical_condition,
               COUNT(*) AS total_patients,
               ROUND(AVG(billing_amount), 2) AS avg_billing
        FROM healthcare
        GROUP BY medical_condition
    """,
    "disease_burden_bmi": """
        SELECT bmi_category,
               COUNT(*) AS patients,
               SUM(heart_disease) AS heart,
               SUM(diabetes) AS diabetes,
               SUM(stroke) AS stroke,
               SUM(cancer) AS cancer
        FROM disease_prediction
        GROUP BY bmi_category
    """,
    "monthly_appointments": """
        SELECT month,
               COUNT(*) AS appointments,
               SUM(status = 'No-Show') AS no_shows
        FROM hospital_management
        GROUP BY month
    """,
    "insurance_summary": """
        SELECT insurance_provider,
               COUNT(*) AS claims,
               ROUND(AVG(billing_amount), 2) AS avg_claim,
               ROUND(SUM(billing_amount), 2) AS total_billed
        FROM healthcare
        GROUP BY insurance_provider
    """,
    "lifestyle_disease": """
        SELECT smoking, alcohol_consumption, exercise,
               COUNT(*) AS patients,
               ROUND(AVG(heart_disease) * 100, 1) AS heart_disease_pct,
               ROUND(AVG(diabetes) * 100, 1)      AS diabetes_pct,
               ROUND(AVG(bmi), 2)                 AS avg_bmi
        FROM disease_prediction
        GROUP BY smoking, alcohol_consumption, exercise
    """,
    "doctor_workload": """
        SELECT h.doctor,
               COUNT(DISTINCT h.patient_id)      AS patients_treated,
               COUNT(DISTINCT hm.appointment_id) AS total_appointments,
               ROUND(AVG(h.billing_amount), 2)   AS avg_billing,
               SUM(hm.status = 'No-Show')        AS no_shows
        FROM healthcare h
        LEFT JOIN hospital_management hm ON h.patient_id = hm.patient_id
        GROUP BY h.doctor
    """
}

# 2. Run and save all standard SQL exports
for filename, sql in exports.items():
    df_export = pd.read_sql(sql, conn)
    path = os.path.join(output_dir, f"{filename}.csv")
    df_export.to_csv(path, index=False)
    print(f"Saved: {filename}.csv  ({len(df_export)} rows)")


# 3. Handle 'patient_risk_combined' separately using Python/Pandas logic
print("\nProcessing patient_risk_combined data...")

df_health_db  = pd.read_sql("SELECT * FROM healthcare", conn).reset_index(drop=True)
df_disease_db = pd.read_sql("SELECT * FROM disease_prediction", conn).reset_index(drop=True)

df_combined = pd.concat([
    df_health_db[['patient_id', 'age', 'gender', 'medical_condition',
                  'admission_type', 'billing_amount', 'test_results']],
    df_disease_db[['bmi', 'bmi_category', 'blood_pressure',
                   'cholesterol', 'smoking', 'exercise',
                   'heart_disease', 'diabetes', 'stroke',
                   'kidney_disease', 'cancer']]
], axis=1)

df_combined['disease_count'] = (
    df_combined['heart_disease'] +
    df_combined['diabetes']      +
    df_combined['stroke']        +
    df_combined['kidney_disease']+
    df_combined['cancer']
)

df_combined['billing_amount'] = df_combined['billing_amount'].round(2)

# Save the combined dataframe
path = os.path.join(output_dir, "patient_risk_combined.csv")
df_combined.to_csv(path, index=False)
print(f"Saved: patient_risk_combined.csv  ({len(df_combined)} rows)")


print("\nAll Power BI exports ready!")

C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\259953948.py:64: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_export = pd.read_sql(sql, conn)
C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\259953948.py:64: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_export = pd.read_sql(sql, conn)
C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\259953948.py:64: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_export = pd.read_sql(sql, conn)
C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780

Saved: billing_by_condition.csv  (6 rows)
Saved: disease_burden_bmi.csv  (3 rows)
Saved: monthly_appointments.csv  (12 rows)
Saved: insurance_summary.csv  (5 rows)
Saved: lifestyle_disease.csv  (8 rows)
Saved: doctor_workload.csv  (40341 rows)

Processing patient_risk_combined data...


C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\259953948.py:73: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_health_db  = pd.read_sql("SELECT * FROM healthcare", conn).reset_index(drop=True)
C:\Users\Ananya Lakkaraju\AppData\Local\Temp\ipykernel_780\259953948.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_disease_db = pd.read_sql("SELECT * FROM disease_prediction", conn).reset_index(drop=True)


Saved: patient_risk_combined.csv  (54966 rows)

All Power BI exports ready!


In [199]:
cursor.close()
conn.close()
print("Connection closed")

Connection closed
